In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import random
import shutil
import pandas as pd

# ============================================================
# SETTINGS (Updated for Google Drive & _top50 folders)
# ============================================================

BASE_DIR = "/content/drive/MyDrive/SafeCue Audio Recordings"
OUTPUT_DIR = os.path.join(BASE_DIR, "split_data")

CLASSES = [
    "Fire Alarm",
    "Smoke Alarm",
    "Siren",
    "Screaming",
    "Background Noise"
]

# Split percentages
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

# Makes the split reproducible
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# ============================================================
# CHECK RATIOS
# ============================================================

if abs(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO - 1.0) > 1e-6:
    raise ValueError("Train, validation, and test ratios must add to 1.")


# ============================================================
# CREATE OUTPUT DIRECTORIES
# ============================================================

for split in ["train", "validation", "test"]:
    for class_name in CLASSES:
        os.makedirs(
            os.path.join(
                OUTPUT_DIR,
                split,
                class_name
            ),
            exist_ok=True
        )


# ============================================================
# STORE INFORMATION ABOUT EVERY FILE
# ============================================================

all_metadata = []


# ============================================================
# PROCESS EACH CLASS
# ============================================================

for class_name in CLASSES:
    # Points directly to your Google Drive sibling folders (e.g., "Fire Alarm_top50")
    class_dir = os.path.join(
        BASE_DIR,
        f"{class_name}_top50"
    )

    if not os.path.exists(class_dir):
        print(f"WARNING: {class_dir} does not exist.")
        continue

    # Get all WAV files in this class folder
    files = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith(".wav")
    ]

    files.sort()
    random.shuffle(files)
    total_files = len(files)

    print(f"\n{class_name}: {total_files} files")


    # ========================================================
    # CALCULATE SPLIT SIZES
    # ========================================================

    train_count = round(total_files * TRAIN_RATIO)
    validation_count = round(total_files * VALIDATION_RATIO)
    test_count = total_files - train_count - validation_count


    # ========================================================
    # SPLIT FILES
    # ========================================================

    train_files = files[:train_count]
    validation_files = files[train_count:train_count + validation_count]
    test_files = files[train_count + validation_count:]

    split_files = {
        "train": train_files,
        "validation": validation_files,
        "test": test_files
    }


    # ========================================================
    # COPY FILES TO GOOGLE DRIVE SPLIT FOLDERS
    # ========================================================

    for split_name, split_list in split_files.items():
        for filename in split_list:
            source_path = os.path.join(class_dir, filename)

            destination_path = os.path.join(
                OUTPUT_DIR,
                split_name,
                class_name,
                filename
            )

            shutil.copy2(source_path, destination_path)

            source_id = os.path.splitext(filename)[0]

            all_metadata.append({
                "filename": filename,
                "source_id": source_id,
                "class": class_name,
                "split": split_name
            })

    print(f"  Train:      {train_count}")
    print(f"  Validation: {validation_count}")
    print(f"  Test:       {test_count}")


# ============================================================
# CREATE DATAFRAME & SAVE METADATA TO DRIVE
# ============================================================

df = pd.DataFrame(all_metadata)

metadata_path = os.path.join(BASE_DIR, "split_metadata.csv")
df.to_csv(metadata_path, index=False)


# ============================================================
# VERIFY RESULTS
# ============================================================

print("\n========================================")
print("SPLIT COMPLETE")
print("========================================")

print(f"Total recordings: {len(df)}")
print(f"Training:         {(df['split'] == 'train').sum()}")
print(f"Validation:       {(df['split'] == 'validation').sum()}")
print(f"Test:             {(df['split'] == 'test').sum()}")

print("\nClass distribution:")
print(
    df.groupby(
        ["split", "class"]
    ).size().unstack(
        fill_value=0
    )
)

print(f"\nMetadata saved to: {metadata_path}")


Fire Alarm: 50 files
  Train:      35
  Validation: 8
  Test:       7

Smoke Alarm: 50 files
  Train:      35
  Validation: 8
  Test:       7

Siren: 50 files
  Train:      35
  Validation: 8
  Test:       7

Screaming: 50 files
  Train:      35
  Validation: 8
  Test:       7

Background Noise: 50 files
  Train:      35
  Validation: 8
  Test:       7

SPLIT COMPLETE
Total recordings: 250
Training:         175
Validation:       40
Test:             35

Class distribution:
class       Background Noise  Fire Alarm  Screaming  Siren  Smoke Alarm
split                                                                  
test                       7           7          7      7            7
train                     35          35         35     35           35
validation                 8           8          8      8            8

Metadata saved to: /content/drive/MyDrive/SafeCue Audio Recordings/split_metadata.csv
